# Libraries and Data

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
movies = pd.read_csv("../data/raw/movies.csv")
print(movies.shape)
movies.head()

(9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



# Feature preparation

In [4]:
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex= False)

In [5]:
movies[["title", "genres_clean"]].head()

,title,genres_clean
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy Romance
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy


## TF-IDF Matrix

In [6]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies["genres_clean"])

print(f"Matrix shape: {tfidf_matrix.shape}")

Matrix shape: (9742, 24)


In [7]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Similarity matrix shape: {cosine_sim.shape}")

Similarity matrix shape: (9742, 9742)


In [10]:
def get_similar_movies(title, n=10):
    # Get the index of the movie
    idx = movies[movies["title"] == title].index[0]

    # Get similarity scores for that movie
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key = lambda x: x[1], reverse = True)

    # Skip the movie itself (index 0)
    scores = [s for s in scores if s[0] != idx][:n]

    movie_indices = [i[0] for i in scores]
    return movies[["title", "genres"]].iloc[movie_indices]

In [11]:
get_similar_movies("Blade Runner (1982)")

,title,genres
59,Lawnmower Man 2: Beyond Cyberspace (1996),Action|Sci-Fi|Thriller
68,Screamers (1995),Action|Sci-Fi|Thriller
144,Johnny Mnemonic (1995),Action|Sci-Fi|Thriller
296,Virtuosity (1995),Action|Sci-Fi|Thriller
336,Timecop (1994),Action|Sci-Fi|Thriller
567,Solo (1996),Action|Sci-Fi|Thriller
601,"Arrival, The (1996)",Action|Sci-Fi|Thriller
939,"Terminator, The (1984)",Action|Sci-Fi|Thriller
1373,Godzilla (1998),Action|Sci-Fi|Thriller
1939,"Matrix, The (1999)",Action|Sci-Fi|Thriller


## Including tags for recommender system

In [12]:
tags = pd.read_csv("../data/raw/tags.csv")

In [13]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [14]:
# Combining all tags per movie into one string
tags_clean = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x)).reset_index()


In [15]:
tags_clean.columns = ["movieId", "tags"]

In [16]:
# Merge tags with movies
movies = movies.merge(tags_clean, on = "movieId", how = "left")
movies["tags"] = movies["tags"].fillna("")

In [17]:
# Combining genres and tags into one string
movies["features"] = movies["genres_clean"] + " " + movies["tags"]
movies[["title", "features"]].head()

,title,features
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy pi...
1,Jumanji (1995),Adventure Children Fantasy fantasy magic board...
2,Grumpier Old Men (1995),Comedy Romance moldy old
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy pregnancy remake


## New tfidf with full features (genres + tags)

In [18]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["features"])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Matrix shape: {tfidf_matrix.shape}")

Matrix shape: (9742, 1677)


In [19]:
get_similar_movies("Blade Runner (1982)")

,title,genres
9604,Blade Runner 2049 (2017),Sci-Fi
8084,Upstream Color (2013),Romance|Sci-Fi|Thriller
9647,The Shape of Water (2017),Adventure|Drama|Fantasy
939,"Terminator, The (1984)",Action|Sci-Fi|Thriller
3352,Short Circuit (1986),Comedy|Sci-Fi
3873,Minority Report (2002),Action|Crime|Mystery|Sci-Fi|Thriller
937,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",Drama
6477,Paprika (Papurika) (2006),Animation|Mystery|Sci-Fi
59,Lawnmower Man 2: Beyond Cyberspace (1996),Action|Sci-Fi|Thriller
68,Screamers (1995),Action|Sci-Fi|Thriller
